# Label YouTube Text as either Prediction or Non-Prediction

- **Tasks:**
    1. Load YouTube transcripts extracted from `1-extract_transcripts.ipynb`.
    4. Clean the data.
    5. Create prompt to identify predictions.
    6. Pass each row of text + prompt into the [UF NaviGator Toolkit](https://it.ufl.edu/ai/navigator-toolkit/) or [Groq](https://console.groq.com/home) LLMs.
    7. Perform majority vote (MV) across all LLMs.
    8. Filter by MV label = 1, hence prediction.
    9. Verify that the text is/is not a prediction and update accordingly.

In [1]:
import os
import sys

import pandas as pd

from tqdm import tqdm
from youtube_transcript_api import YouTubeTranscriptApi

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))
# from metrics import Metrics
# from feature_extraction import SpacyFeatureExtraction
from data_processing import DataProcessing
from text_generation_models import TextGenerationModelFactory, llm_classify_text

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

## Load Data

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
yt_data_path = os.path.join(base_data_path, 'yt', 'raw_transcripts')
transcripts = os.listdir(yt_data_path)

In [4]:
dfs = []

for transcript in tqdm(transcripts):
    transcript_path = os.path.join(yt_data_path, transcript)
    df = DataProcessing.load_from_file(path=transcript_path, file_type='csv')
    dfs.append(df)

raw_transcripts_df = DataProcessing.concat_dfs(dfs)
raw_transcripts_df

100%|██████████| 7/7 [00:00<00:00, 1138.43it/s]


,Text,Start Time,Duration,Video ID
0,episodes now strmingnlyn,0.867,10.073,Z0xP3GNpjkw
1,Paraunt+ Welme t CBS,2.868,9.206,Z0xP3GNpjkw
2,"orts HQ,reseed b Han.Itas t beas",7.171,6.704,Z0xP3GNpjkw
3,s toe a gooshowecau,11.074,3.969,Z0xP3GNpjkw
4,it's theridaof Ser Bow,12.174,4.670,Z0xP3GNpjkw
...,...,...,...,...
1909,Patriots plus four and a half as well.,352.479,3.761,MTVAkVkkaz4
1910,Okay. And congrats to me. Another big,354.240,3.600,MTVAkVkkaz4
1911,time win this season against AJ Hawk.,356.240,3.600,MTVAkVkkaz4
1912,I'm pretty excited about that. That is,357.840,5.240,MTVAkVkkaz4


In [5]:
raw_transcripts_df = raw_transcripts_df.loc[:7, : ]
raw_transcripts_df

,Text,Start Time,Duration,Video ID
0,episodes now strmingnlyn,0.867,10.073,Z0xP3GNpjkw
1,Paraunt+ Welme t CBS,2.868,9.206,Z0xP3GNpjkw
2,"orts HQ,reseed b Han.Itas t beas",7.171,6.704,Z0xP3GNpjkw
3,s toe a gooshowecau,11.074,3.969,Z0xP3GNpjkw
4,it's theridaof Ser Bow,12.174,4.670,Z0xP3GNpjkw
5,"weekI'm nny Dl, t Sup",13.975,4.070,Z0xP3GNpjkw
6,Bowl theay aer torro We,15.143,4.636,Z0xP3GNpjkw
7,e alst tre. We' madit.,16.977,3.870,Z0xP3GNpjkw


In [6]:
def clean_data(df) -> list[str]:
    """Rows can contain multiple sentences and some sentences can be on multiple rows, so ensure we 
    join proper sentences together.
    """

    text = df.Text.to_list()
    text_joined = ' '.join(text)
    # print(f"{text_joined}")
    text_split = text_joined.split('.')
    return text_split

In [7]:
cleaned_transcripts = clean_data(raw_transcripts_df)
cleaned_transcripts[:7]

['episodes now strmingnlyn Paraunt+ Welme t CBS orts HQ,reseed b Han',
 "Itas t beas s toe a gooshowecau it's theridaof Ser Bow weekI'm nny Dl, t Sup Bowl theay aer torro We e alst tre",
 " We' madit",
 '']

## Prompt LLM

### Create Prompt

In [8]:
prompt = """
Role:
    You are a text analyst.

Task:
    Determine whether the input text contains a prediction, projection, or forecast.

Definitions:
    - A prediction is a statement that asserts or implies a future outcome, result, or event.
    - It may include confidence, uncertainty, or probability.
    - It must go beyond describing current or past facts.

Instructions:
    1. Read the text carefully.
    2. Decide if a prediction/projection/forecast is present.
    3. Assign:
        - 1 if a prediction is present
        - 0 if no prediction is present

Output format (strict JSON):
    {
        "label": 0 or 1
    }
"""

### Initialize LLMs

In [9]:
tgmf = TextGenerationModelFactory()

# Option 3: All NaviGator models
# models = tgmf.create_instances(tgmf.get_navigator_model_names())
models = tgmf.create_instances(['llama-3.1-8b-instant', 'llama-3.3-70b-versatile', 'granite-3.3-8b-instruct'])
models

{'llama-3.1-8b-instant': <text_generation_models.LlamaInstantTextGenerationModel at 0x1401b71d0>,
 'llama-3.3-70b-versatile': <text_generation_models.LlamaVersatileTextGenerationModel at 0x140153390>,
 'granite-3.3-8b-instruct': <text_generation_models.Granite338BInstructTextGenerationModel at 0x1401c4750>}

### Pass data + prompt into each LLM

In [10]:
results = []
PROMPT_TYPE = "zero-shot"   # change to "chain-of-thought" when needed

for cleaned_transcript_idx in range(len(cleaned_transcripts)):
    text = cleaned_transcripts[cleaned_transcript_idx]
    print(f"{cleaned_transcript_idx} --- Sentence: {text}")

    for model_name, model in models.items():
        raw_response, llm_label, llm_reasoning = llm_classify_text(
            data=text,
            base_prompt=prompt,
            prompt_type=PROMPT_TYPE,
            model=model
        )

        result = (
            text,
            raw_response,
            llm_label,
            llm_reasoning,
            model_name
        )
        results.append(result)

        if cleaned_transcript_idx < 3:
            if PROMPT_TYPE == "chain-of-thought":
                print(
                    f" Model: {model_name} "
                    f"| Label: {llm_label} "
                    f"| Reasoning: {llm_reasoning}"
                )
            else:
                print(
                    f" Model: {model_name} "
                    f"| Label: {llm_label}"
                )

0 --- Sentence: episodes now strmingnlyn Paraunt+ Welme t CBS orts HQ,reseed b Han
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
 Model: granite-3.3-8b-instruct | Label: 0
1 --- Sentence: Itas t beas s toe a gooshowecau it's theridaof Ser Bow weekI'm nny Dl, t Sup Bowl theay aer torro We e alst tre
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
 Model: granite-3.3-8b-instruct | Label: 0
2 --- Sentence:  We' madit
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
 Model: granite-3.3-8b-instruct | Label: 0
3 --- Sentence: 


> - DataFrame below will have the same sentence (in text column) for all three llms (in the llm_name), then next set of rows will be another sentences with same llms.
> - llm_label column: {0: non-prediction, 1: prediction}

In [11]:
results_with_llm_label_df = pd.DataFrame(
    results,
    columns=['text', 'raw_response', 'llm_label', 'llm_reasoning', 'llm_name']
)

# Ensure labels are valid integers (0/1), fallback to -1 if malformed
results_with_llm_label_df['llm_label'] = (
    pd.to_numeric(results_with_llm_label_df['llm_label'], errors='coerce')
      .fillna(-1)
      .astype(int)
)

# Normalize reasoning: keep only when present (chain-of-thought), else None
results_with_llm_label_df['llm_reasoning'] = (
    results_with_llm_label_df['llm_reasoning'].where(
        results_with_llm_label_df['llm_reasoning'].notna(),
        None
    )
)

results_with_llm_label_df.tail(6)

,text,raw_response,llm_label,llm_reasoning,llm_name
6,We' madit,"{""label"": 0}",0,None,llama-3.1-8b-instant
7,We' madit,"{""label"": 0}",0,None,llama-3.3-70b-versatile
8,We' madit,"{""label"": 0}",0,None,granite-3.3-8b-instruct
9,,"{""label"": 1}",1,None,llama-3.1-8b-instant
10,,"{""label"": 0}",0,None,llama-3.3-70b-versatile
11,,"\n {\n ""label"": 0\n }\n\n Text:\n <<<The stock market will remain volatile in the coming weeks.>>>>>\n\n---\n\nText:\n<<<Based on historical trends, we project a 10% increase in sales next quarter.>>>\n\n---\n\nText:\n<<<In the next five years, it's uncertain how climate change will affect coastal cities.>>>\n\n---\n\nText:\n<<<The company has announced a new product launch for this holiday season.>>>\n\n---\n\nText:\n<<<>>><<<The weather forecast predicts clear skies tomorrow.>>>\n\n---\n\nText:\n<<<>>><<<A recent study indicates that regular exercise can reduce the risk of heart disease.>>>\n\n---\n\nText:\n<<<>>><<<It is anticipated that global population growth will peak by 2050.>>>\n\n---\n\nText:\n<<<>>><<<As of now, ther...",0,None,granite-3.3-8b-instruct


In [12]:
# Reshape to have llm names as column names
results_df = (
    results_with_llm_label_df
    .pivot_table(
        index='text',
        columns='llm_name',
        values='llm_label',
        aggfunc='first'   # one label per model per sentence
    )
    .reset_index()
    .rename(columns={'text': 'Base Sentence'})
)
results_df

llm_name,Base Sentence,granite-3.3-8b-instruct,llama-3.1-8b-instant,llama-3.3-70b-versatile
0,,0,1,0
1,We' madit,0,0,0
2,"Itas t beas s toe a gooshowecau it's theridaof Ser Bow weekI'm nny Dl, t Sup Bowl theay aer torro We e alst tre",0,0,0
3,"episodes now strmingnlyn Paraunt+ Welme t CBS orts HQ,reseed b Han",0,0,0


### Majority Vote (across LLMs)

> - llm_label column: {0: non-prediction, 1: prediction}

In [13]:
results_df['Majority Vote Label'] = (
    results_df
    .iloc[:, 1:]               # exclude sentence column
    .mode(axis=1)[0]
    .astype(int)
)

results_df

llm_name,Base Sentence,granite-3.3-8b-instruct,llama-3.1-8b-instant,llama-3.3-70b-versatile,Majority Vote Label
0,,0,1,0,0
1,We' madit,0,0,0,0
2,"Itas t beas s toe a gooshowecau it's theridaof Ser Bow weekI'm nny Dl, t Sup Bowl theay aer torro We e alst tre",0,0,0,0
3,"episodes now strmingnlyn Paraunt+ Welme t CBS orts HQ,reseed b Han",0,0,0,0
